# DVS event-only motion compensation on DSEC

This notebook validates the historical constant-translation baseline for **DVS event-image motion compensation**. The frozen default is BA in `local_affine.ipynb`; the integrated B/BA/BH comparison is in `homography.ipynb`. This baseline uses the repository notebook for reproduction; a separate executed Colab copy is not retained.

Goals:

1. Run entirely on a free Colab CPU runtime.
2. Read real DSEC events and calibration without requiring a camera or RK3588.
3. Estimate a local constant event displacement by maximizing accumulated event-image variance.
4. Compare that estimate with DSEC forward optical flow in the same patch.
5. Expose the main limitation: one translation cannot represent a full driving frame containing rotation, parallax, and moving objects.

## Method

For an event window `[t0, t1]`, events are split into temporal bins. We search a constant endpoint displacement `v = (vx, vy)` and warp each bin to the midpoint `t_ref`:

```text
x' = x - vx * (tau - 0.5)
y' = y - vy * (tau - 0.5)
```

The objective is the variance of the polarity-weighted, lightly smoothed accumulated event image. DSEC optical flow is used only for evaluation, not inside the optimizer.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/itsvvy/dsec-event-motion-compensation.git"
BRANCH = "main"
REPO_DIR = Path("/content/dsec-event-motion-compensation")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/dsec-event-motion-compensation
sys.path.insert(0, str(REPO_DIR))
%pip install -q -r requirements.txt

## 1. Sanity check with known motion

The synthetic test seeds a spatial event pattern with a known translation. This verifies the warp direction and search independently of DSEC.

In [ ]:
from tests.test_synthetic import test_recovers_known_constant_translation

test_recovers_known_constant_translation()
print("Synthetic constant-translation test passed.")

## 2. Run DSEC on CPU

The first run downloads and extracts the smallest practical DSEC sequence (`thun_00_a`, left events, about 285 MB) and the matching 11 MB forward-flow labels. Three 100 ms windows and three spatial patches are evaluated.

In [ ]:
%env MPLCONFIGDIR=/content/.matplotlib
%time !python scripts/run_dsec_demo.py --data-dir /content/dsec_data --output-dir /content/dsec_results --flow-indices 2 8 12

## 3. Quantitative and qualitative results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

results_dir = Path("/content/dsec_results")
summary = json.loads((results_dir / "summary.json").read_text())
table = pd.DataFrame(summary["results"])
columns = [
    "flow_index", "patch_center_yx", "patch_event_count",
    "estimated_vx_px", "estimated_vy_px", "gt_vx_px", "gt_vy_px",
    "endpoint_error_px", "variance_gain", "gradient_energy_gain", "runtime_s"
]
display(table[columns].round(3))

print("median endpoint error:", round(summary["median_endpoint_error_px"], 3), "px")
print("median variance gain:", round(summary["median_variance_gain"], 3), "x")

for figure in sorted(results_dir.glob("dsec_f*.png"))[:6]:
    print(figure.name)
    display(Image(filename=str(figure), width=1200))

## Interpretation

The result supports a staged feasibility claim:

- Event-only motion compensation runs on CPU and visibly refocuses event contours.
- In local regions with one dominant motion, the estimated displacement is close to DSEC optical flow.
- A full-frame constant translation is not sufficient for driving scenes. The deliberate left-side failure patch shows that variance may increase while the selected motion differs from the patch median.

This verification therefore de-risks the data path and baseline algorithm before buying or wiring the RK3588/DVS hardware, but it does not yet establish real-time embedded performance. The next implementation step should be block-wise translation or local affine/homography estimation, followed by 10-50 ms CPU timing tests and RK3588 porting.